# Business Statistics — SQ2
**Amanda Luijendijk**
This is the same SQ2 section as in the shared group notebook.


### Import the analysis libraries

Loads pandas for tables, NumPy for numerical operations, Matplotlib for charts, SciPy for statistical functions, and charset-normalizer for encoding utilities. Some imports are retained from Fred’s notebook even though later cells do not use them.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

### Load the survey
Use BS_DATA_PATH for a local file, or the shared Responses.xlsx in Google Drive. The raw file is never changed.


In [ ]:
import os
from pathlib import Path
if 'google.colab' in __import__('sys').modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
candidates = [Path(os.environ.get('BS_DATA_PATH', 'Responses.xlsx')),
              Path('/content/drive/MyDrive/Colab Notebooks/Responses.xlsx'),
              Path.home() / 'Downloads' / 'Responses (3).xlsx']
file_path = next((p for p in candidates if p.is_file()), None)
if file_path is None:
    raise FileNotFoundError('Place Responses.xlsx beside this notebook or set BS_DATA_PATH.')
df = pd.read_excel(file_path)
for col in df.select_dtypes(include=['object', 'string']):
    df[col] = df[col].map(lambda v: v.strip() if isinstance(v, str) else v)
print('Loaded survey:', df.shape)


# SQ2 — Amanda Luijendijk
## My contribution: personalisation awareness, control and age

**SQ2:** How is awareness of feed personalisation related to perceived control, and does this relationship differ across age groups?

**What this section covers:** checking and coding the SQ2 data, tables, a chart, a regression with age groups, a rank correlation, an age interaction test, and the conclusion. Supporting work below includes ranking and scaling, the earlier age-group test, and the model with social media use.

**What do we measure?** This question focuses on awareness. The survey does not ask how much personalisation a person receives. We use this statement: “I am informed that my feed is shaped by personal information like my age, gender or location”. We call this **awareness**. It measures what people know about feed shaping. It does not directly measure personalisation.

We measure **control** with “I feel I am in control of my feed”. Both answers run from 1 (Strongly disagree) to 5 (Strongly agree).

**Our test question:** Is awareness linked to control when we include age group? Does this link differ between age groups?

**Main H0:** There is no awareness–control link in the model with age groups. **Main H1:** There is a link.

**Age-interaction H0:** The awareness–control slope is the same across age groups. **Age-interaction H1:** At least one age group has a different slope.

We use regression with age as categories. This treats the five answer steps as equally spaced. HC3 gives standard errors that allow different amounts of variation. We also use Spearman correlation, which works with the order of answers. A separate interaction test checks whether the link differs by age. Small groups make that test less reliable.

This is survey data. A link does not prove cause and effect. This contribution was prepared with Codex assistance. Amanda Luijendijk should review the code and be able to explain the choices.


In [ ]:
import statsmodels.formula.api as smf
from IPython.display import display, Markdown

agreement_scale = {'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly agree': 5}
sq2_columns = {
    'awareness': 'I am informed that my feed is shaped by personal information like my age, gender or location - transparency\xa0',
    'control': 'I feel I am in control of my feed',
}
sq2_all = pd.DataFrame(index=df.index)
for short, header in sq2_columns.items():
    answers = df[header]
    unexpected = set(answers.dropna()) - set(agreement_scale)
    if unexpected:
        raise ValueError(f'Unexpected response labels in {short}: {unexpected}')
    sq2_all[short] = answers.map(agreement_scale).astype(float)
sq2_all['age_group'] = df['What age group are you in?']
age_order = ['Under 18', '18-24', '25-34', '35-44', '45-54', '55-64', '65 and older']
unknown_ages = set(sq2_all.age_group.dropna()) - set(age_order)
if unknown_ages:
    raise ValueError(f'Unknown age bands: {unknown_ages}')
print('Missing values (before complete-case selection):')
display(sq2_all.isna().sum().to_frame('missing'))
sq2 = sq2_all.dropna().copy()
print('Included:', len(sq2), '| Excluded:', len(df) - len(sq2))
print('Repeated non-missing respondent IDs:', df['ID'].dropna().duplicated().sum())
# This check cannot establish independent sampling.
sq2_summary = sq2.groupby('age_group')[['awareness', 'control']].agg(['count', 'mean', 'median', 'std'])
display(sq2_summary.reindex(age_order).round(3))


## Amanda Luijendijk — Compare answers by age
Each bar shows the percentages within one age group. A score of 1 means strongly disagree; 5 means strongly agree. Groups with no answers are not plotted. The table above keeps all seven age bands.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ct = pd.crosstab(sq2['age_group'], sq2['control']).reindex(age_order).dropna(how='all')
ct = ct.reindex(columns=[1, 2, 3, 4, 5], fill_value=0)
ct.div(ct.sum(axis=1), axis=0).mul(100).plot.bar(stacked=True, ax=ax)
ax.set(xlabel='Age group', ylabel='Within-group percentage', title='SQ2: perceived control by age group')
ax.legend(title='Control score (1–5)', bbox_to_anchor=(1.02, 1))
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()


## Amanda Luijendijk — Three-category chi-square test before SEM
**Question:** Are awareness of feed personalisation and perceived control associated?

We group the original five answers into three categories: **Disagree** = strongly disagree or disagree (1–2); **Neutral** = 3; **Agree** = agree or strongly agree (4–5). These groups have a clear meaning. This loses some detail, so the regression and SEM keep the original five-point scores. This extra test was added after the earlier analyses and is exploratory.

**H0:** Awareness category and control category are independent (no association). **H1:** They are associated. We use a significance level of 0.05.

The table contains counts, not percentages. Each respondent contributes once. We check expected counts. If any expected count is below 1, or more than 20% are below 5, we use a permutation p-value for the Pearson chi-square statistic. The permutation keeps the row and column totals fixed and uses 9,999 random shuffles with a fixed seed. We do not merge more categories to obtain a significant result.

This test does not account for age, use the order of the categories, show a direction, or prove cause and effect. It is different from the SEM chi-square model-fit test. Different sample sizes are possible because this test needs only the two answers; SEM needs all its own variables.


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from IPython.display import display, Markdown

# Keep the source unchanged. Use the two questions needed for this test.
chi_source = df.copy()
chi_source.columns = chi_source.columns.str.replace('\u00a0', ' ', regex=False).str.strip()
chi_headers = {
    'awareness': 'I am informed that my feed is shaped by personal information like my age, gender or location - transparency',
    'control': 'I feel I am in control of my feed',
}
chi_scale = {'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly agree': 5}
chi_three = {1: 'Disagree', 2: 'Disagree', 3: 'Neutral', 4: 'Agree', 5: 'Agree'}
chi_data = pd.DataFrame(index=chi_source.index)
for short, header in chi_headers.items():
    answers = chi_source[header].astype('string').str.strip().replace('', pd.NA)
    if not answers.dropna().isin(chi_scale).all():
        raise ValueError(f'Unknown response labels in {short}')
    chi_data[short] = answers.map(chi_scale).map(chi_three)
chi_data = chi_data.dropna()
chi_order = ['Disagree', 'Neutral', 'Agree']
chi_table = pd.crosstab(chi_data['awareness'], chi_data['control']).reindex(index=chi_order, columns=chi_order, fill_value=0)
# Remove a category only if nobody selected it; this prevents zero margins.
chi_observed = chi_table.loc[chi_table.sum(axis=1) > 0, chi_table.sum(axis=0) > 0]
if min(chi_observed.shape) < 2:
    raise ValueError('At least two observed categories are needed for each variable.')
chi_stat, chi_p, chi_df, chi_expected = chi2_contingency(chi_observed, correction=False)
chi_sparse = bool((chi_expected < 1).any() or (chi_expected < 5).mean() > .20)
chi_final_p = float(chi_p)
chi_method = 'Pearson chi-square approximation'
if chi_sparse:
    rng = np.random.default_rng(20261102)
    a = pd.Categorical(chi_data['awareness'], categories=chi_observed.index).codes
    b = pd.Categorical(chi_data['control'], categories=chi_observed.columns).codes
    nr, nc = chi_observed.shape
    extreme = 0
    chi_resamples = 9999
    for _ in range(chi_resamples):
        shuffled = np.bincount(a * nc + rng.permutation(b), minlength=nr * nc).reshape(nr, nc)
        perm_stat = ((shuffled - chi_expected) ** 2 / chi_expected).sum()
        extreme += perm_stat >= chi_stat - 1e-12
    chi_final_p = (extreme + 1) / (chi_resamples + 1)
    chi_method = 'permutation p-value (9,999 shuffles)'
chi_v = np.sqrt(chi_stat / (len(chi_data) * min(chi_observed.shape[0]-1, chi_observed.shape[1]-1)))
print('Observed counts: awareness in rows, control in columns')
display(chi_table)
print('Expected counts under H0:')
display(pd.DataFrame(chi_expected, index=chi_observed.index, columns=chi_observed.columns).round(2))
print('Included:', len(chi_data), '| Excluded:', len(chi_source) - len(chi_data))
print('Minimum expected count:', round(float(chi_expected.min()), 2))
print('Expected counts below 5:', int((chi_expected < 5).sum()), 'of', chi_expected.size)
chi_decision = 'We reject H0. There is evidence of an association between the three-category answers.' if chi_final_p < .05 else 'We do not reject H0. We did not find clear evidence of an association between the three-category answers.'
display(Markdown(f"""### Amanda Luijendijk — Chi-square result
**N = {len(chi_data)}**, Pearson **χ²({chi_df}) = {chi_stat:.3f}**. The ordinary approximate p-value is **{chi_p:.6g}**. We use **{chi_method}**, giving **p = {chi_final_p:.6g}**. Cramér's V = **{chi_v:.3f}** (association size, without a direction).

{chi_decision}

This is an exploratory, unadjusted test. It does not account for age or replace the main regression and age-interaction analysis. Its p-value is not corrected for all the tests in the project. Agree the final set of tests with the group before making confirmatory claims. The SEM model-fit chi-square asks a different question.
"""))


### Amanda Luijendijk — Simple view of the chi square result
These bars show the percentage of control answers within each awareness category. The chi square test compares all cells together; the chart does not show significant pairwise differences.


In [ ]:
import matplotlib.pyplot as plt
chi_pct = chi_table.div(chi_table.sum(axis=1), axis=0).mul(100)
ax = chi_pct.plot.bar(figsize=(7, 4), color=['#9aa6b2', '#d2a85a', '#4c8b91'], rot=0)
ax.set(xlabel='Awareness category', ylabel='Percentage within category', ylim=(0, 100), title=f'Awareness and control: chi-square p = {chi_final_p:.3f}')
ax.legend(title='Control', ncol=3, fontsize=9)
plt.tight_layout(); plt.show()


## Amanda Luijendijk — Test the link between awareness and control
The regression includes awareness and age group. The awareness coefficient shows the expected change in the control score for one step higher awareness, while accounting for age group.

The 95% confidence interval shows uncertainty around the estimate. HC3 allows unequal variation. It does not solve problems with the survey questions or sampling. Spearman correlation is an extra check using ranks; it does not account for age.

The residual chart shows the gaps between the model's predictions and the real scores. Stripes are expected because answers only have five values. We do not need to make raw answer scores normally distributed.


In [ ]:
sq2_model = smf.ols('control ~ awareness + C(age_group)', data=sq2).fit(cov_type='HC3')
X = sq2_model.model.exog
if np.linalg.matrix_rank(X) < X.shape[1]:
    raise ValueError('Design matrix is not full rank.')
sq2_results = pd.DataFrame({'coefficient': sq2_model.params, 'p': sq2_model.pvalues})
sq2_results[['CI_low', 'CI_high']] = sq2_model.conf_int()
display(sq2_results.round(5))
rho, rho_p = stats.spearmanr(sq2['awareness'], sq2['control'])
print(f'Unadjusted Spearman rho={rho:.4f}, p={rho_p:.6g}')
print(f'N={int(sq2_model.nobs)}, R-squared={sq2_model.rsquared:.4f}')
# Residual check: a discrete outcome naturally produces bands.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(sq2_model.fittedvalues, sq2_model.resid, alpha=.2)
ax.axhline(0, color='black')
ax.set(xlabel='Fitted control score', ylabel='Residual', title='SQ2: residual check')
plt.tight_layout()
plt.show()


### Amanda Luijendijk — Simple view of the main awareness result
The dot is the estimated awareness coefficient. The line shows its 95% HC3 confidence interval. The dashed zero line means no linear awareness association in this model.


In [ ]:
import matplotlib.pyplot as plt
viz_b = float(sq2_model.params['awareness'])
viz_lo, viz_hi = sq2_model.conf_int().loc['awareness']
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.errorbar([viz_b], [0], xerr=[[viz_b-viz_lo], [viz_hi-viz_b]], fmt='o', color='steelblue', capsize=6)
ax.axvline(0, color='grey', linestyle='--')
ax.set(yticks=[0], yticklabels=['Awareness'], xlabel='Control score difference per awareness step', title=f'Main regression: p = {float(sq2_model.pvalues["awareness"]):.3f}')
ax.set_ylim(-.7, .7); plt.tight_layout(); plt.show()


## Amanda Luijendijk — Does the link differ by age?
The interaction model allows a different awareness–control link in each age group. One joint test checks these differences together. This is an extra, exploratory test. Some age groups are very small, so use care when discussing the result.

The table also shows a rank correlation for each group. These numbers describe patterns. They do not show which pairs of groups differ significantly.


In [ ]:
interaction_model = smf.ols('control ~ awareness * C(age_group)', data=sq2).fit(cov_type='HC3')
interaction_terms = [i for i, name in enumerate(interaction_model.params.index) if ':' in name]
R = np.eye(len(interaction_model.params))[interaction_terms]
interaction_test = interaction_model.wald_test(R, scalar=True)
interaction_p = float(interaction_test.pvalue)
print(f'Joint interaction Wald statistic={float(interaction_test.statistic):.4f}, df={len(interaction_terms)}, p={interaction_p:.6g}')
# Age-specific rank correlations are descriptive; no claims from multiple unadjusted tests.
rows = []
for age, group in sq2.groupby('age_group'):
    r = stats.spearmanr(group.awareness, group.control).statistic if group.awareness.nunique() > 1 and group.control.nunique() > 1 else np.nan
    rows.append({'age_group': age, 'n': len(group), 'Spearman_rho': r})
display(pd.DataFrame(rows).set_index('age_group').reindex(age_order).round(3))


### Amanda Luijendijk — Simple view of the link within each age group
Each dot shows the fitted awareness slope for one age group, with a 95% HC3 interval. Small groups have uncertain estimates. The joint interaction test compares slopes together; these intervals are not corrected for multiple comparisons.


In [ ]:
import matplotlib.pyplot as plt
from patsy import build_design_matrices
viz_slopes = []
for viz_age in age_order:
    viz_n = int((sq2['age_group'] == viz_age).sum())
    if not viz_n: continue
    viz_rows = pd.DataFrame({'awareness': [1., 2.], 'age_group': [viz_age, viz_age]})
    viz_design = np.asarray(build_design_matrices([interaction_model.model.data.design_info], viz_rows)[0])
    viz_test = interaction_model.t_test(viz_design[1] - viz_design[0])
    viz_effect = float(np.asarray(viz_test.effect).item())
    viz_low, viz_high = np.asarray(viz_test.conf_int()).ravel()
    viz_slopes.append((viz_age, viz_n, viz_effect, viz_low, viz_high))
fig, ax = plt.subplots(figsize=(8, 4))
for viz_i, (viz_age, viz_n, viz_effect, viz_low, viz_high) in enumerate(viz_slopes):
    ax.errorbar(viz_effect, viz_i, xerr=[[viz_effect-viz_low], [viz_high-viz_effect]], fmt='o', capsize=4, color='steelblue')
ax.axvline(0, linestyle='--', color='grey')
ax.set(yticks=range(len(viz_slopes)), yticklabels=[f'{a} (n={n})' for a,n,*_ in viz_slopes], xlabel='Control score difference per awareness step', title=f'Exploratory age interaction: joint p = {interaction_p:.5f}')
ax.invert_yaxis(); plt.tight_layout(); plt.show()


## Amanda Luijendijk — Main conclusion
The next cell writes the conclusion from the results. It also shows a temporary Bonferroni correction for three planned group questions: multiply p by three, up to a maximum of one. The group still needs to agree which tests belong in the final correction. This does not correct all the extra tests in this notebook.


In [ ]:
b = float(sq2_model.params['awareness'])
lo, hi = sq2_model.conf_int().loc['awareness']
p = float(sq2_model.pvalues['awareness'])
p3 = min(1., 3 * p)
main_text = 'We found evidence of a link after accounting for age.' if p3 < .05 else 'We did not find clear evidence of an overall link after accounting for age.'
age_text = 'The extra test suggests that the link differs between age groups.' if interaction_p < .05 else 'The extra test did not find clear evidence that the link differs between age groups.'
display(Markdown(f"""### Amanda Luijendijk — Results in simple English
We used **{len(sq2)} complete responses** out of **{len(df)}**. {main_text}

For one step higher awareness, the model estimated **{b:.3f}** points difference in control. The 95% HC3 confidence interval was **{lo:.3f} to {hi:.3f}**. The p-value was **{p:.6g}**; the temporary three-question corrected p-value was **{p3:.6g}**. R² was **{sq2_model.rsquared:.3f}**, so the model explains only a small part of the differences in control.

The extra rank check gave Spearman rho = **{rho:.3f}**, p = **{rho_p:.6g}**. This is the link before accounting for age.

{age_text} Its p-value was **{interaction_p:.6g}**. Small groups make this exploratory result less secure. This test asks a different question from the overall model, so the two results can differ.

**Answer to SQ2:** We did not find clear evidence of an overall awareness–control link after accounting for age. The exploratory interaction test suggests that this link differs between age groups. Small groups mean we should treat that result with care. This study measures awareness, not actual personalisation, and does not show cause and effect.

**Limits:** We use one question for each main measure. Some age groups are small and unequal in size. There are no respondents aged 55–64. We leave out rows with missing required answers. We assume independent responses, but checking duplicate IDs cannot prove this.
"""))


## Amanda Luijendijk — Supporting work: ranks and scaling
This keeps the data-preparation work from the earlier SQ2 notebook. Ranks put answers in order. Min–max scaling puts them between 0 and 1. Z-scores show distance from the mean in standard deviations. Tied answers get average ranks. Missing answers stay missing. Scaling does not make the answers normally distributed. The main model above uses the original 1–5 scores.


In [ ]:
sq2_scores = sq2_all[['awareness', 'control']]
sq2_ranks = sq2_scores.rank(method='average')
sq2_minmax = (sq2_scores - sq2_scores.min()) / (sq2_scores.max() - sq2_scores.min()).replace(0, np.nan)
sq2_zscores = (sq2_scores - sq2_scores.mean()) / sq2_scores.std(ddof=0).replace(0, np.nan)
# Show only summary statistics, not individual survey answers.
for label, table in [('Ranks', sq2_ranks), ('Min–max scores', sq2_minmax), ('Z-scores', sq2_zscores)]:
    print(label)
    display(table.describe().round(3))


### Amanda Luijendijk — Simple view of ranking and scaling
Each point maps an original answer category to its average rank, min max score or z score. These are alternative ways to express the same ordered answers; they are not extra hypothesis tests.


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for viz_ax, (viz_name, viz_values) in zip(axes, [('Average rank', sq2_ranks), ('Min max score', sq2_minmax), ('Z score', sq2_zscores)]):
    for viz_col in ['awareness', 'control']:
        viz_map = pd.DataFrame({'original': sq2_scores[viz_col], 'transformed': viz_values[viz_col]}).dropna().groupby('original')['transformed'].mean()
        viz_ax.plot(viz_map.index, viz_map.values, marker='o', label=viz_col.title())
    viz_ax.set(xlabel='Original answer (1–5)', ylabel=viz_name, xticks=[1,2,3,4,5], title=viz_name)
axes[0].legend(fontsize=9); plt.tight_layout(); plt.show()


## Amanda Luijendijk — Supporting work: the earlier age question
**Question:** Do control scores differ between age groups? This supports SQ2, but does not test personalisation.

**H0:** The distributions of control scores are the same across age groups. **H1:** At least one differs.

Kruskal–Wallis compares ranked answers across independent groups. It does not require normally distributed answers. It compares distributions; it is only a test of medians when the distributions have similar shapes. It does not tell us which pairs differ. This test needs age and control only, so its sample can differ from the main model.


In [ ]:
sq2_age = sq2_all[['age_group', 'control']].dropna()
age_samples = [g['control'].to_numpy() for _, g in sq2_age.groupby('age_group')]
age_h, age_p = stats.kruskal(*age_samples)
display(sq2_age.groupby('age_group')['control'].agg(['count', 'mean', 'median']).reindex(age_order).round(3))
display(Markdown(f"With **{len(sq2_age)} complete answers**, Kruskal–Wallis H({len(age_samples)-1}) = **{age_h:.3f}**, p = **{age_p:.6g}**. The temporary three-question corrected p-value is **{min(age_p*3,1):.6g}**. " + ('There is evidence that control-score distributions differ by age. This does not show which groups differ.' if age_p < .05 else 'There is no clear evidence of different distributions.')))


### Amanda Luijendijk — Simple view of the age comparison
Bars show mean control scores and sample sizes. Means treat the response steps as equally spaced. The Kruskal Wallis test compares distributions, so the bars alone do not identify statistically different pairs.


In [ ]:
import matplotlib.pyplot as plt
viz_age_summary = sq2_age.groupby('age_group')['control'].agg(['count', 'mean']).reindex(age_order).dropna()
fig, ax = plt.subplots(figsize=(8, 4))
viz_bars = ax.bar(viz_age_summary.index, viz_age_summary['mean'], color='steelblue')
ax.bar_label(viz_bars, labels=[f'n={int(v)}' for v in viz_age_summary['count']], padding=4)
ax.set(ylim=(0, 5), xlabel='Age group', ylabel='Mean control (1–5)', title=f'Control by age: Kruskal–Wallis p = {age_p:.6f}')
plt.xticks(rotation=25); plt.tight_layout(); plt.show()


## Amanda Luijendijk — Supporting work: include social media use
This is the earlier extra model. It asks whether age is linked to control when we also include social media use frequency. Both predictors are categories. It does not test personalisation, and is not the main answer to the updated SQ2.

HC3 allows unequal variation. The model still assumes equally spaced control scores and independent responses. The joint age test checks all age terms together. R² shows how much variation the model explains. These are extra analyses, not independent confirmation of the main result.


In [ ]:
sq2_use = sq2_all[['control', 'age_group']].copy()
sq2_use['social_use'] = df['I use social media']
sq2_use = sq2_use.dropna()
use_model = smf.ols('control ~ C(age_group) + C(social_use)', data=sq2_use).fit(cov_type='HC3')
use_x = use_model.model.exog
assert np.linalg.matrix_rank(use_x) == use_x.shape[1], 'Model columns overlap.'
use_table = pd.DataFrame({'coefficient': use_model.params, 'p': use_model.pvalues})
use_table[['CI_low', 'CI_high']] = use_model.conf_int()
display(use_table.round(4))
age_terms = [i for i, name in enumerate(use_model.params.index) if name.startswith('C(age_group)')]
use_test = use_model.wald_test(np.eye(len(use_model.params))[age_terms], scalar=True)
display(Markdown(f"The model used **{len(sq2_use)} responses**. The joint age test gave p = **{float(use_test.pvalue):.6g}**. R² = **{use_model.rsquared:.3f}**. " + ('Age is linked to control after including use frequency.' if float(use_test.pvalue) < .05 else 'We did not find clear evidence of an age link after including use frequency.') + ' This does not show that age causes control.'))


### Amanda Luijendijk — Simple view of age after including social media use
Dots show differences from age 18–24 after accounting for use frequency. Lines show unadjusted 95% HC3 intervals. The reported joint age p value tests all age terms together.


In [ ]:
import matplotlib.pyplot as plt
viz_ci = use_model.conf_int()
fig, ax = plt.subplots(figsize=(8, 4))
for viz_i, viz_j in enumerate(age_terms):
    viz_name = use_model.params.index[viz_j]
    viz_value = float(use_model.params.iloc[viz_j]); viz_low, viz_high = viz_ci.loc[viz_name]
    ax.errorbar(viz_value, viz_i, xerr=[[viz_value-viz_low], [viz_high-viz_value]], fmt='o', capsize=4, color='steelblue')
ax.set(yticks=range(len(age_terms)), yticklabels=[use_model.params.index[j].split('[T.')[-1].rstrip(']') for j in age_terms], xlabel='Control score difference versus age 18–24', title=f'Age and use frequency: joint age p = {float(use_test.pvalue):.4f}')
ax.axvline(0, linestyle='--', color='grey'); ax.invert_yaxis(); plt.tight_layout(); plt.show()


## Amanda Luijendijk — How to explain my section
- **What did you study?** Awareness of feed personalisation, perceived control and age.
- **Why use awareness?** The survey has no direct measure of how much personalisation people receive. This limits the answer.
- **Why regression?** To include awareness and age in the same model.
- **Why use a rank check?** The answers have an order, but equal gaps between them are not certain.
- **Why test an interaction?** To check whether the awareness–control link differs by age.
- **What did the supporting work add?** Ranking and scaling, an age-group comparison, and an extra model with social media use.
- **What can you conclude?** Describe the observed links and uncertainty. Do not claim cause and effect.

**Earlier SEM work:** The separate SEM notebook on GitHub is an earlier exploratory approach. It does not answer the updated personalisation question. It is not used as evidence for this section because the control scale was weak and the model needs a separate review.

**Contribution note:** This section is assigned to Amanda Luijendijk and was prepared with Codex assistance. It covers the work listed above. The earlier shared sections belong to the group; they are not claimed as Amanda's individual work.
